# 📊 Mutual Fund Analytics
## Day 2: AUM by Fund House — Data Cleaning

### 🎯 Objective
The objective of this notebook is to inspect, clean, and validate Assets Under Management (AUM) data for different mutual fund houses.

### 📁 Dataset Information
- **Source:** `data/raw/03_aum_by_fund_house.csv`
- **Output:** `data/processed/cleaned_aum_by_fund_house.csv`

### 🛠️ Tools & Technologies
- Python
- Pandas
- Jupyter Notebook

### 🧹 Cleaning Workflow
1. Load and inspect the raw dataset.
2. Check column names and data types.
3. Identify missing values and duplicates.
4. Validate AUM values and other relevant fields.
5. Apply appropriate cleaning rules.
6. Verify the cleaned dataset.
7. Export the processed CSV.

### ✅ Expected Outcome
A validated AUM dataset ready for further analysis and integration into the Mutual Fund Analytics database.

In [1]:
import pandas as pd
from pathlib import Path

project_path = Path.cwd().parent

aum_file = (
    project_path
    / "data"
    / "raw"
    / "03_aum_by_fund_house.csv"
)

aum_df = pd.read_csv(aum_file)

print("Dataset shape:", aum_df.shape)
print("\nColumn names:")
print(aum_df.columns.tolist())

display(aum_df.head())

Dataset shape: (90, 5)

Column names:
['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']


,date,fund_house,aum_lakh_crore,aum_crore,num_schemes
0,2022-03-31,SBI Mutual Fund,6.05,605000,186
1,2022-03-31,ICICI Prudential MF,4.65,465000,216
2,2022-03-31,HDFC Mutual Fund,4.35,435000,195
3,2022-03-31,Nippon India MF,2.70,270000,177
4,2022-03-31,Kotak Mahindra MF,2.70,270000,168


In [2]:
# 1. Check data types
print("DATA TYPES")
print(aum_df.dtypes)

# 2. Check missing values
print("\nMISSING VALUES")
print(aum_df.isnull().sum())

# 3. Check duplicate rows
print("\nDUPLICATE ROWS")
print("Complete duplicate rows:", aum_df.duplicated().sum())

# 4. Check duplicate date and fund-house combinations
print("\nDUPLICATE DATE-FUND HOUSE COMBINATIONS")
print(
    aum_df.duplicated(
        subset=["date", "fund_house"]
    ).sum()
)

# 5. Check unique values
print("\nUNIQUE VALUES")
print("Unique dates:", aum_df["date"].nunique())
print("Unique fund houses:", aum_df["fund_house"].nunique())

# 6. Inspect numeric columns
numeric_columns = [
    "aum_lakh_crore",
    "aum_crore",
    "num_schemes"
]

print("\nNUMERIC COLUMN SUMMARY")
display(aum_df[numeric_columns].describe())

# 7. Check invalid numeric values
print("\nNON-POSITIVE VALUES")

for column in numeric_columns:
    print(
        f"{column}:",
        (aum_df[column] <= 0).sum()
    )

# 8. Inspect date range
aum_dates = pd.to_datetime(aum_df["date"], errors="coerce")

print("\nDATE VALIDATION")
print("Invalid dates:", aum_dates.isna().sum())
print("Earliest date:", aum_dates.min())
print("Latest date:", aum_dates.max())

DATA TYPES
date                  str
fund_house            str
aum_lakh_crore    float64
aum_crore           int64
num_schemes         int64
dtype: object

MISSING VALUES
date              0
fund_house        0
aum_lakh_crore    0
aum_crore         0
num_schemes       0
dtype: int64

DUPLICATE ROWS
Complete duplicate rows: 0

DUPLICATE DATE-FUND HOUSE COMBINATIONS
0

UNIQUE VALUES
Unique dates: 9
Unique fund houses: 10

NUMERIC COLUMN SUMMARY


,aum_lakh_crore,aum_crore,num_schemes
count,90.000000,9.000000e+01,90.000000
mean,4.352889,4.352889e+05,152.200000
std,2.734328,2.734328e+05,52.108832
min,1.050000,1.050000e+05,56.000000
25%,2.525000,2.525000e+05,95.000000
50%,3.450000,3.450000e+05,172.500000
75%,5.675000,5.675000e+05,195.000000
max,12.500000,1.250000e+06,216.000000



NON-POSITIVE VALUES
aum_lakh_crore: 0
aum_crore: 0
num_schemes: 0

DATE VALIDATION
Invalid dates: 0
Earliest date: 2022-03-31 00:00:00
Latest date: 2025-12-31 00:00:00


In [3]:
# 1. Check whether AUM values are mathematically consistent
expected_aum_crore = aum_df["aum_lakh_crore"] * 100000

aum_difference = (
    aum_df["aum_crore"] - expected_aum_crore
).abs()

print("AUM conversion mismatches:", (aum_difference > 0.01).sum())

# 2. Check whitespace in fund house names
print(
    "\nFund house names with leading/trailing spaces:",
    (
        aum_df["fund_house"]
        != aum_df["fund_house"].str.strip()
    ).sum()
)

# 3. Inspect the fund house names
print("\nFund house names:")
print(sorted(aum_df["fund_house"].unique()))

# 4. Check chronological order
aum_dates = pd.to_datetime(aum_df["date"], errors="coerce")

print(
    "\nDates already sorted:",
    aum_dates.is_monotonic_increasing
)

# 5. Check number of fund houses per date
fund_houses_per_date = aum_df.groupby("date")["fund_house"].nunique()

print("\nFund houses per date:")
print(fund_houses_per_date.to_string())

print(
    "\nDates with other than 10 fund houses:",
    (fund_houses_per_date != 10).sum()
)

AUM conversion mismatches: 0

Fund house names with leading/trailing spaces: 0

Fund house names:
['Aditya Birla Sun Life MF', 'Axis Mutual Fund', 'DSP Mutual Fund', 'HDFC Mutual Fund', 'ICICI Prudential MF', 'Kotak Mahindra MF', 'Mirae Asset MF', 'Nippon India MF', 'SBI Mutual Fund', 'UTI Mutual Fund']

Dates already sorted: True

Fund houses per date:
date
2022-03-31    10
2022-09-30    10
2023-03-31    10
2023-09-30    10
2024-03-31    10
2024-09-30    10
2024-12-31    10
2025-03-31    10
2025-12-31    10

Dates with other than 10 fund houses: 0


In [4]:
# Create a copy of the original dataset
aum_clean = aum_df.copy()

# Convert date column to datetime
aum_clean["date"] = pd.to_datetime(
    aum_clean["date"],
    errors="coerce"
)

# Sort records by date and fund house
aum_clean = aum_clean.sort_values(
    by=["date", "fund_house"]
).reset_index(drop=True)

# Validate the cleaned dataset
print("Cleaned dataset shape:", aum_clean.shape)
print("Missing values:", aum_clean.isnull().sum().sum())
print("Duplicate rows:", aum_clean.duplicated().sum())
print(
    "Duplicate date-fund house combinations:",
    aum_clean.duplicated(
        subset=["date", "fund_house"]
    ).sum()
)
print("Date dtype:", aum_clean["date"].dtype)

Cleaned dataset shape: (90, 5)
Missing values: 0
Duplicate rows: 0
Duplicate date-fund house combinations: 0
Date dtype: datetime64[us]


In [5]:
# Define output path
output_file = (
    project_path
    / "data"
    / "processed"
    / "cleaned_aum_by_fund_house.csv"
)

# Create processed folder if it doesn't exist
output_file.parent.mkdir(parents=True, exist_ok=True)

# Export cleaned dataset
aum_clean.to_csv(output_file, index=False)

print("AUM dataset exported successfully!")
print("Saved location:", output_file)
print("Exported rows:", len(aum_clean))

AUM dataset exported successfully!
Saved location: d:\Mutual-Fund-Analytics\data\processed\cleaned_aum_by_fund_house.csv
Exported rows: 90


In [6]:
# Reload the exported dataset
aum_check = pd.read_csv(
    output_file,
    parse_dates=["date"]
)

# Verify the exported file
print("Exported shape:", aum_check.shape)
print("Missing values:", aum_check.isnull().sum().sum())
print("Duplicate rows:", aum_check.duplicated().sum())

print(
    "Duplicate date-fund house combinations:",
    aum_check.duplicated(
        subset=["date", "fund_house"]
    ).sum()
)

print("Date dtype:", aum_check["date"].dtype)

Exported shape: (90, 5)
Missing values: 0
Duplicate rows: 0
Duplicate date-fund house combinations: 0
Date dtype: datetime64[us]


## ✅ Final Cleaning Summary

### Dataset Overview
- **Source:** `data/raw/03_aum_by_fund_house.csv`
- **Output:** `data/processed/cleaned_aum_by_fund_house.csv`
- **Final dimensions:** 90 rows × 5 columns

### Data Quality Results
- Missing values: 0
- Duplicate rows: 0
- Duplicate date–fund house combinations: 0
- Invalid dates: 0
- AUM conversion mismatches: 0
- All 10 fund houses are present across all 9 reporting dates.

### Transformations Applied
- Converted `date` to datetime format.
- Sorted records by date and fund house.
- Exported the cleaned dataset without an additional index column.
- Reloaded the exported CSV for verification.

### Final Outcome
The AUM by Fund House dataset passed the validation checks and is ready for further analysis in the Mutual Fund Analytics project.